### ALBERT for Paraphrase detection

In this code, ALBERT model will be fine-tuned for paraphrase detection task. This is the final project for the course "TDT4310 - Intelligent Text Analytics and Language Understanding" at NTNU. The implementation of this project were carried out by Jens Christian Valen Leynse and Vincenzo Spinello

### Install the packages

In [ ]:
!pip install transformers torch scikit-learn
!pip install pyunpack
!pip install patool


Define the libraries

In [ ]:
import torch
from transformers import AlbertTokenizer, AlbertForSequenceClassification, Trainer, TrainingArguments, AutoModel
from sklearn.model_selection import train_test_split
from torch.optim import Adam, SGD
import numpy as np
import torch.nn as nn
import pyunpack
from pyunpack import Archive
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
from tqdm import tqdm

Download the dataset

In [ ]:
# Repository link: https://github.com/vinc-00/ParaphraseDetectionData.git

!git clone https://github.com/vinc-00/ParaphraseDetectionData.git

fatal: destination path 'ParaphraseDetectionData' already exists and is not an empty directory.


Check the dataset

In [ ]:
%cd /content/ParaphraseDetectionData
!ls

/content/ParaphraseDetectionData
test_1.csv.zip	train.csv  train.csv.zip


Extract the data

In [ ]:
Archive('/content/ParaphraseDetectionData/train.csv.zip').extractall('/content/ParaphraseDetectionData/') #extract the dataset

Open the dataset and print the head and the length

In [ ]:
# Training dataset dir
dataset_dir = '/content/ParaphraseDetectionData/train.csv'
dataset = pd.read_csv(dataset_dir)
print(dataset.head())
print('length of the dataset: ', len(dataset))

   id  qid1  qid2                                          question1  \
0   0     1     2  What is the step by step guide to invest in sh...   
1   1     3     4  What is the story of Kohinoor (Koh-i-Noor) Dia...   
2   2     5     6  How can I increase the speed of my internet co...   
3   3     7     8  Why am I mentally very lonely? How can I solve...   
4   4     9    10  Which one dissolve in water quikly sugar, salt...   

                                           question2  is_duplicate  
0  What is the step by step guide to invest in sh...             0  
1  What would happen if the Indian government sto...             0  
2  How can Internet speed be increased by hacking...             0  
3  Find the remainder when [math]23^{24}[/math] i...             0  
4            Which fish would survive in salt water?             0  
length of the dataset:  404290


## Preprocess the data

In [ ]:
#################  sentence1, sentence2, label (1 paraphrase, 0 not)

dataset = dataset[['question1', 'question2', 'is_duplicate']]
print(dataset.head())

# sampled_dataset = dataset.sample(n=20000)

# Split the dataset into train, validation and test
train_data, temp_data = train_test_split(dataset, test_size=0.2)

# Split temp set into validation and test sets (50% validation, 50% test)
val_data, test_data = train_test_split(temp_data, test_size=0.5)


                                           question1  \
0  What is the step by step guide to invest in sh...   
1  What is the story of Kohinoor (Koh-i-Noor) Dia...   
2  How can I increase the speed of my internet co...   
3  Why am I mentally very lonely? How can I solve...   
4  Which one dissolve in water quikly sugar, salt...   

                                           question2  is_duplicate  
0  What is the step by step guide to invest in sh...             0  
1  What would happen if the Indian government sto...             0  
2  How can Internet speed be increased by hacking...             0  
3  Find the remainder when [math]23^{24}[/math] i...             0  
4            Which fish would survive in salt water?             0  


## Define the CustomDataset Class

In [ ]:
class PDataset(Dataset):

  def split(self, dataset):
    self.sentences1 = dataset[['question1']]
    self.sentences2 = dataset[['question2']]
    if self.with_target: self.targets = dataset[['is_duplicate']]
    self.sentences1.reset_index(drop=True, inplace=True)
    self.sentences2.reset_index(drop=True, inplace=True)
    if self.with_target: self.targets.reset_index(drop=True, inplace=True)


  def __init__(self, dataset, model='albert-base-v2', maxlen=128, with_target=True):
    self.with_target = with_target
    self.split(dataset)
    self.tokenizer = AutoTokenizer.from_pretrained(model)
    self.maxlen = maxlen


  def __getitem__(self, index):
    sentence1 = str(self.sentences1.loc[index])
    sentence2 = str(self.sentences2.loc[index])
    if self.with_target: target = torch.tensor(self.targets.loc[index], dtype=torch.float32)

    encoded_sample = self.tokenizer(sentence1, sentence2, padding='max_length', truncation=True, max_length=self.maxlen, return_tensors='pt')

    token_ids = encoded_sample['input_ids'].squeeze(0)
    attn_masks = encoded_sample['attention_mask'].squeeze(0)
    token_type_ids = encoded_sample['token_type_ids'].squeeze(0)

    if self.with_target:
      return token_ids, attn_masks, token_type_ids, target
    else:
      return token_ids, attn_masks, token_type_ids

  #this is another mandatory method; it returns the length of the dataset
  def __len__(self):
    return len(self.sentences1)

## Define the model

In [ ]:
import torch.nn.init as init

class ALBERTClassifier(nn.Module):

    def __init__(self, bert_model="albert-base-v2", freeze_bert=True):
        super(ALBERTClassifier, self).__init__()

        self.albert_layer = AutoModel.from_pretrained(bert_model)

        if bert_model == "albert-base-v2":  # 12M parameters
            hidden_size = 768
        elif bert_model == "albert-large-v2":  # 18M parameters
            hidden_size = 1024
        elif bert_model == "albert-xlarge-v2":  # 60M parameters
            hidden_size = 2048

        # Freeze albert layers and only train the classification layer weights
        if freeze_bert:
            for p in self.albert_layer.parameters():
                p.requires_grad = False

        # self.cls_layer1 = nn.Linear(hidden_size, 256)
        # init.xavier_uniform_(self.cls_layer1.weight)
        # init.xavier_uniform_(self.cls_layer1.weight)

        # self.cls_layer2 = nn.Linear(256, 1)
        # init.xavier_uniform_(self.cls_layer2.weight)
        # init.constant_(self.cls_layer2.bias, 0)

        # Classification layer
        self.cls_layer = nn.Linear(hidden_size, 1)

        #init.xavier_uniform_(self.cls_layer.weight)
        #init.constant_(self.cls_layer.bias, 0)

        self.dropout = nn.Dropout(p=0.2)


    def forward(self, input_ids, attn_masks, token_type_ids):


        # Feeding the inputs to the BERT-based model to obtain contextualized representations
        pooler_output = self.albert_layer(input_ids, attn_masks, token_type_ids)['pooler_output']
        # cont_reps, pooler_output = self.albert_layer(input_ids, attn_masks, token_type_ids)

        logits = self.cls_layer(self.dropout(pooler_output))

        # probabilities = torch.sigmoid(logits)

        # hidden_output = self.dropout(torch.relu(self.cls_layer1(pooler_output)))

        # logits = self.cls_layer2(hidden_output)

        # probabilities = torch.sigmoid(logits)

        return logits

## Define the training procedure

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=5, device='cuda'):
    # Move model to device
    model.to(device)

    train_losses = []
    train_accuracies = []
    val_losses = []
    val_accuracies = []

    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0
        total_predictions = 0
        correct_predictions = 0

        # Iterate over training batches
        for batch in tqdm(train_loader, desc=f'Epoch {epoch + 1}'):
            inputs, masks, token_types, targets = batch
            inputs, masks, token_types, targets = inputs.to(device), masks.to(device), token_types.to(device), targets.to(device)


            # Zero the gradients
            optimizer.zero_grad()

            # Forward pass
            outputs = model(inputs, masks, token_types)

            # Calculate loss

            loss = criterion(outputs, targets)

            # Backward pass
            loss.backward()

            # Update weights
            optimizer.step()

            # Accumulate epoch loss
            epoch_loss += loss.item()

            # Calculate the accuracy
            predicted_labels = (outputs > 0.5).float()
            correct_predictions += (predicted_labels == targets).sum().item()
            total_predictions += targets.size(0)

        # Calculate average epoch loss
        avg_epoch_loss = epoch_loss / len(train_loader)

        #Calculate the training accuracy
        train_accuracy = correct_predictions / total_predictions

        # Append training loss and accuracy
        train_losses.append(avg_epoch_loss)
        train_accuracies.append(train_accuracy)

        # Print training loss for the epoch
        print(f"Epoch {epoch + 1}, Train Loss: {avg_epoch_loss:.4f}, Train Accuracy: {train_accuracy:.4f}")

        # Validate the model
        val_loss, val_acc = evaluate_model(model, val_loader, criterion, device=device)

        val_losses.append(val_loss)
        val_accuracies.append(val_acc)

        print(f"Epoch {epoch + 1}, Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_acc:.4f}")

    print("Training complete.")
    return train_losses, train_accuracies, val_losses, val_accuracies

## Define the evaluation method

In [ ]:
def evaluate_model(model, data_loader, criterion, device='cuda'):
    model.eval()
    total_loss = 0.0
    num_samples = 0
    total_predictions = 0
    correct_predictions = 0

    with torch.no_grad():
        for batch in data_loader:
            inputs, masks, token_types, targets = batch
            inputs, masks, token_types, targets = inputs.to(device), masks.to(device), token_types.to(device), targets.to(device)

            outputs = model(inputs, masks, token_types)
            loss = criterion(outputs, targets)

            # Calculate the loss
            total_loss += loss.item() * inputs.size(0)
            num_samples += inputs.size(0)

            # Calculate the accuracy
            predicted_labels = (outputs > 0.5).float()
            correct_predictions += (predicted_labels == targets).sum().item()
            total_predictions += targets.size(0)

    return total_loss / num_samples, correct_predictions / total_predictions

## Define the parameters for the model

In [ ]:
albert_model = "albert-base-v2"  # 'albert-base-v2', 'albert-large-v2', 'albert-xlarge-v2', 'albert-xxlarge-v2', 'bert-base-uncased'
freeze_bert = True
maxlen = 128  # maximum length of the tokenized input sentence pair : if greater than "maxlen", the input is truncated and else if smaller, the input is padded
bs = 32  # batch size
learning_rate = 0.0005  # learning rate
epochs = 5  # number of training epochs

## Instantiate the dataloaders

In [ ]:
dataloader_train = DataLoader(PDataset(train_data, albert_model, maxlen, with_target=True), batch_size=32, shuffle=False)
dataloader_val = DataLoader(PDataset(val_data, albert_model, maxlen, with_target=True), batch_size=32, shuffle=False)
dataloader_test = DataLoader(PDataset(test_data, albert_model, maxlen, with_target=True), batch_size=32, shuffle=False)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/760k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.31M [00:00<?, ?B/s]

## Instantiate the model and its parameters

In [ ]:
albert_v2 = ALBERTClassifier()

criterion = nn.BCELoss()
optimizer = Adam(albert_v2.parameters(), lr=learning_rate)

model.safetensors:   0%|          | 0.00/47.4M [00:00<?, ?B/s]

## Train the model

In [ ]:
train_losses_v2, train_accuracies_v2, val_losses_v2, val_accuracies_v2 = train_model(albert_v2, dataloader_train, dataloader_val, criterion, optimizer, epochs, device='cuda')

Epoch 1: 100%|██████████| 5000/5000 [23:56<00:00,  3.48it/s]


Epoch 1, Train Loss: 0.5785, Train Accuracy: 0.6763
Epoch 1, Validation Loss: 0.5413, Validation Accuracy: 0.7029


Epoch 2: 100%|██████████| 5000/5000 [23:55<00:00,  3.48it/s]


Epoch 2, Train Loss: 0.5455, Train Accuracy: 0.7029
Epoch 2, Validation Loss: 0.5265, Validation Accuracy: 0.7207


Epoch 3: 100%|██████████| 5000/5000 [23:55<00:00,  3.48it/s]


Epoch 3, Train Loss: 0.5371, Train Accuracy: 0.7131
Epoch 3, Validation Loss: 0.5224, Validation Accuracy: 0.7217


Epoch 4: 100%|██████████| 5000/5000 [23:54<00:00,  3.48it/s]


Epoch 4, Train Loss: 0.5316, Train Accuracy: 0.7166
Epoch 4, Validation Loss: 0.5229, Validation Accuracy: 0.7255


Epoch 5: 100%|██████████| 5000/5000 [23:55<00:00,  3.48it/s]


Epoch 5, Train Loss: 0.5274, Train Accuracy: 0.7206
Epoch 5, Validation Loss: 0.5245, Validation Accuracy: 0.7217
Training complete.


## Define the test procedure

In [ ]:
def test_model(model, test_loader, criterion, device='cuda'):
    # Move model to device
    model.to(device)

    # Switch to evaluation mode
    model.eval()

    # Initialize variables for evaluation
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    # Disable gradient calculation
    with torch.no_grad():
        for batch in tqdm(test_loader, desc='Testing'):
            inputs, masks, token_types, targets = batch
            inputs, masks, token_types, targets = inputs.to(device), masks.to(device), token_types.to(device), targets.to(device)

            # Forward pass
            outputs = model(inputs, masks, token_types)

            # Calculate loss
            loss = criterion(outputs, targets)

            # Accumulate total loss
            total_loss += loss.item()

            # Calculate predictions
            predicted_labels = (outputs > 0.5).float()
            total_correct += (predicted_labels == targets).sum().item()
            total_samples += targets.size(0)

    # Calculate average loss
    avg_loss = total_loss / len(test_loader)

    # Calculate accuracy
    accuracy = total_correct / total_samples

    print(f"Test Loss: {avg_loss:.4f}, Accuracy: {accuracy:.4f}")
    return avg_loss, accuracy

## Test the model

In [ ]:
test_loss_v2, test_acc_v2,  = test_model(albert_v2, dataloader_test, criterion)

Testing: 100%|██████████| 625/625 [02:57<00:00,  3.52it/s]

Test Loss: 0.5278, Accuracy: 0.7184


## Save the model

In [ ]:
torch.save(albert_v2.state_dict(), 'albert_v2.pth')

## Plot the results

In [ ]:
import matplotlib.pyplot as plt

def plot_results(train_losses, train_accuracies, val_losses, val_accuracies):
    epochs = range(1, len(train_losses) + 1)

    # Plot losses
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.plot(epochs, train_losses, label='Train')
    plt.plot(epochs, val_losses, label='Validation')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.title('Training and Validation Losses')
    plt.legend()

    # Plot accuracies
    plt.subplot(1, 2, 2)
    plt.plot(epochs, train_accuracies, label='Train')
    plt.plot(epochs, val_accuracies, label='Validation')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.title('Training and Validation Accuracies')
    plt.legend()

    plt.tight_layout()
    plt.show()


plot_results(train_losses_v2, train_accuracies_v2, val_losses_v2, val_accuracies_v2)

## Train Albert 2

In [ ]:
albert_2 = ALBERTClassifier()
learning_rate = 0.0005
epochs = 2

criterion = nn.BCEWithLogitsLoss()
optimizer = Adam(albert_2.parameters(), lr=learning_rate)

In [ ]:
train_losses_large, train_accuracies_large, val_losses_large, val_accuracies_large = train_model(albert_2, dataloader_train, dataloader_val, criterion, optimizer, epochs, device='cuda')

Epoch 1: 100%|██████████| 10108/10108 [46:54<00:00,  3.59it/s]


Epoch 1, Train Loss: 0.5628, Train Accuracy: 0.6700
Epoch 1, Validation Loss: 0.5223, Validation Accuracy: 0.7114


Epoch 2: 100%|██████████| 10108/10108 [46:48<00:00,  3.60it/s]


Epoch 2, Train Loss: 0.5353, Train Accuracy: 0.6919
Epoch 2, Validation Loss: 0.5134, Validation Accuracy: 0.7173
Training complete.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

torch.save(albert_2.state_dict(), '/content/drive/My Drive/ALBERT_2.pth')
drive.flush_and_unmount()

In [ ]:
test_model(albert_2, dataloader_test, criterion)

Testing: 100%|██████████| 1264/1264 [05:50<00:00,  3.61it/s]

Test Loss: 0.5136, Accuracy: 0.7163


(0.5135844441578735, 0.7162927601474189)

ALBERTLAR

In [ ]:
albert_model = "albert-large-v2"  # 'albert-base-v2', 'albert-large-v2', 'albert-xlarge-v2', 'albert-xxlarge-v2', 'bert-base-uncased'
freeze_bert = True
maxlen = 128  # maximum length of the tokenized input sentence pair : if greater than "maxlen", the input is truncated and else if smaller, the input is padded
bs = 16  # batch size
learning_rate = 0.0005  # learning rate
epochs = 2  # number of training epochs

dataloader_train = DataLoader(PDataset(train_data, albert_model, maxlen, with_target=True), batch_size=32, shuffle=False)
dataloader_val = DataLoader(PDataset(val_data, albert_model, maxlen, with_target=True), batch_size=32, shuffle=False)
dataloader_test = DataLoader(PDataset(test_data, albert_model, maxlen, with_target=True), batch_size=32, shuffle=False)

In [ ]:
albert_l = ALBERTClassifier(bert_model=albert_model)

criterion = nn.BCEWithLogitsLoss()
optimizer = Adam(albert_l.parameters(), lr=learning_rate)

In [ ]:
train_losses_v2, train_accuracies_v2, val_losses_v2, val_accuracies_v2 = train_model(albert_l, dataloader_train, dataloader_val, criterion, optimizer, epochs, device='cuda')

Epoch 1: 100%|██████████| 500/500 [06:55<00:00,  1.20it/s]


Epoch 1, Train Loss: 0.6420, Train Accuracy: 0.6324
Epoch 1, Validation Loss: 0.6239, Validation Accuracy: 0.6485


Epoch 2: 100%|██████████| 500/500 [07:06<00:00,  1.17it/s]


Epoch 2, Train Loss: 0.6322, Train Accuracy: 0.6317
Epoch 2, Validation Loss: 0.6147, Validation Accuracy: 0.6495
Training complete.


Test another model

### SVM

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.feature_extraction.text import TfidfVectorizer

# Replace missing values with empty string
val_data['question1'].fillna('', inplace=True)
val_data['question2'].fillna('', inplace=True)
train_data['question1'].fillna('', inplace=True)
train_data['question2'].fillna('', inplace=True)

# Combine training and validation data for fitting TF-IDF vectorizer
combined_questions = train_data['question1'] + " " + train_data['question2']
combined_questions = pd.concat([combined_questions, val_data['question1'] + " " + val_data['question2']])

# Initialize labels for training and validation data
y_train = train_data['is_duplicate']
y_val = val_data['is_duplicate']

# Initialize and fit the TF-IDF vectorizer on the combined data
tfidf_vectorizer = TfidfVectorizer()
X_combined_tfidf = tfidf_vectorizer.fit_transform(combined_questions)

# Transform training and validation data
X_train_tfidf = X_combined_tfidf[:len(train_data)]
X_val_tfidf = X_combined_tfidf[len(train_data):len(train_data)+len(val_data)]

# Initialize SVM model
svm_model = SVC(kernel='linear')

# Train the SVM model
svm_model.fit(X_train_tfidf, y_train)

# Evaluate the SVM model on the validation set
y_val_pred = svm_model.predict(X_val_tfidf)
accuracy = accuracy_score(y_val, y_val_pred)
precision = precision_score(y_val, y_val_pred)
recall = recall_score(y_val, y_val_pred)
f1 = f1_score(y_val, y_val_pred)

print("Validation Accuracy:", accuracy)
print("Validation Precision:", precision)
print("Validation Recall:", recall)
print("Validation F1 Score:", f1)